# 分布式訓練與多GPU並行
:label:`sec_distributed_training`

## 概述

隨著模型規模的增長，單GPU訓練已經無法滿足需求。本章將介紹：

- 🚀 數據並行 (DataParallel, DistributedDataParallel)
- 🔄 模型並行 (Pipeline Parallelism, Tensor Parallelism)
- 🌐 多節點訓練
- ⚡ 混合精度訓練
- 🤖 AI輔助優化工具

## 為什麼需要分布式訓練？

- **更大的模型**: GPT-3 (175B parameters), GPT-4 (1.7T+ parameters)
- **更大的數據集**: ImageNet-21K, LAION-5B
- **更快的訓練速度**: 線性或近線性加速比

## 1. 單機多GPU訓練

### 1.1 DataParallel (簡單但有局限性)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 定義一個簡單的模型
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# 檢查GPU數量
device_count = torch.cuda.device_count()
print(f"可用GPU數量: {device_count}")

if device_count > 1:
    # 創建模型
    model = SimpleModel()
    
    # 使用DataParallel包裝模型
    model = nn.DataParallel(model)
    model = model.cuda()
    
    print(f"✅ 模型已分配到 {device_count} 個GPU")
    print(f"DataParallel device_ids: {model.device_ids}")
else:
    print("⚠️ 只有一個GPU可用，DataParallel不會有加速效果")
    model = SimpleModel().cuda()

In [ ]:
# 訓練循環
def train_with_dataparallel(model, dataloader, epochs=3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(dataloader):
            inputs, labels = inputs.cuda(), labels.cuda()
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}], Loss: {running_loss/100:.4f}")
                running_loss = 0.0

# 創建假數據
fake_data = torch.randn(10000, 784)
fake_labels = torch.randint(0, 10, (10000,))
dataset = TensorDataset(fake_data, fake_labels)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# 訓練
if device_count > 1:
    print("\n開始使用 DataParallel 訓練...")
    train_with_dataparallel(model, dataloader, epochs=2)

### 1.2 DistributedDataParallel (推薦使用) ⭐

DistributedDataParallel (DDP) 比 DataParallel 更高效：
- 每個GPU有獨立的進程
- 通過all-reduce高效同步梯度
- 支持多節點訓練

In [ ]:
# 保存以下代碼到 train_ddp.py
ddp_train_code = '''
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler

def setup(rank, world_size):
    """初始化分布式訓練環境"""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    
    # 初始化進程組
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup():
    """清理分布式訓練環境"""
    dist.destroy_process_group()

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

def train_ddp(rank, world_size):
    """DDP訓練函數"""
    print(f"Running DDP on rank {rank}")
    setup(rank, world_size)
    
    # 創建模型並移到GPU
    model = SimpleModel().to(rank)
    
    # 使用DDP包裝模型
    ddp_model = DDP(model, device_ids=[rank])
    
    # 創建數據集和分布式採樣器
    fake_data = torch.randn(10000, 784)
    fake_labels = torch.randint(0, 10, (10000,))
    dataset = TensorDataset(fake_data, fake_labels)
    
    sampler = DistributedSampler(
        dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=128,
        sampler=sampler,
        num_workers=2
    )
    
    # 優化器和損失函數
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(ddp_model.parameters(), lr=0.001)
    
    # 訓練循環
    ddp_model.train()
    for epoch in range(3):
        sampler.set_epoch(epoch)  # 確保每個epoch的數據順序不同
        
        for i, (inputs, labels) in enumerate(dataloader):
            inputs, labels = inputs.to(rank), labels.to(rank)
            
            optimizer.zero_grad()
            outputs = ddp_model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            if rank == 0 and (i + 1) % 50 == 0:
                print(f"Epoch [{epoch+1}/3], Step [{i+1}], Loss: {loss.item():.4f}")
    
    cleanup()

def main():
    world_size = torch.cuda.device_count()
    print(f"Training on {world_size} GPUs")
    
    # 使用torch.multiprocessing啟動多個進程
    import torch.multiprocessing as mp
    mp.spawn(
        train_ddp,
        args=(world_size,),
        nprocs=world_size,
        join=True
    )

if __name__ == "__main__":
    main()
'''

# 保存代碼
with open('train_ddp.py', 'w') as f:
    f.write(ddp_train_code)

print("✅ DDP training code saved to train_ddp.py")
print("\n運行命令:")
print("  python train_ddp.py")
print("\n或使用torchrun (推薦):")
print("  torchrun --nproc_per_node=2 train_ddp.py")

## 2. 混合精度訓練

混合精度訓練使用float16進行計算，float32存儲主權重，可以：
- 減少內存使用 (~50%)
- 加快訓練速度 (~2-3x)
- 在NVIDIA Tensor Core上獲得更好性能

In [ ]:
from torch.cuda.amp import autocast, GradScaler

def train_with_amp(model, dataloader, epochs=3):
    """使用自動混合精度訓練"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scaler = GradScaler()  # 梯度縮放器
    
    model.train()
    for epoch in range(epochs):
        for i, (inputs, labels) in enumerate(dataloader):
            inputs, labels = inputs.cuda(), labels.cuda()
            
            optimizer.zero_grad()
            
            # 自動混合精度上下文
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            # 縮放損失並反向傳播
            scaler.scale(loss).backward()
            
            # 更新權重
            scaler.step(optimizer)
            scaler.update()
            
            if (i + 1) % 100 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}], Loss: {loss.item():.4f}")

# 創建模型並訓練
model = SimpleModel().cuda()
print("\n開始使用混合精度訓練...")
# train_with_amp(model, dataloader, epochs=2)

## 3. 模型並行

### 3.1 張量並行 (Tensor Parallelism)

將單個層的參數分割到多個GPU

In [ ]:
# 簡單的張量並行示例
class TensorParallelLinear(nn.Module):
    def __init__(self, in_features, out_features, num_gpus=2):
        super().__init__()
        self.num_gpus = num_gpus
        
        # 在每個GPU上創建一部分權重
        self.linears = nn.ModuleList([
            nn.Linear(in_features, out_features // num_gpus).to(f'cuda:{i}')
            for i in range(num_gpus)
        ])
    
    def forward(self, x):
        # 將輸入複製到每個GPU
        outputs = []
        for i, linear in enumerate(self.linears):
            x_gpu = x.to(f'cuda:{i}')
            outputs.append(linear(x_gpu))
        
        # 將結果連接起來
        output = torch.cat([o.to('cuda:0') for o in outputs], dim=-1)
        return output

if torch.cuda.device_count() >= 2:
    tp_layer = TensorParallelLinear(784, 512, num_gpus=2)
    test_input = torch.randn(32, 784).cuda()
    output = tp_layer(test_input)
    print(f"✅ 張量並行層輸出形狀: {output.shape}")
else:
    print("⚠️ 需要至少2個GPU才能演示張量並行")

### 3.2 流水線並行 (Pipeline Parallelism)

將模型的不同層放置在不同的GPU上

In [ ]:
class PipelineParallelModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 將不同的層放在不同的GPU上
        self.layer1 = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU()
        ).to('cuda:0')
        
        self.layer2 = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU()
        ).to('cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0')
        
        self.layer3 = nn.Linear(256, 10).to(
            'cuda:0' if torch.cuda.device_count() > 1 else 'cuda:0'
        )
    
    def forward(self, x):
        # Layer 1 on GPU 0
        x = self.layer1(x.to('cuda:0'))
        
        # Layer 2 on GPU 1
        x = self.layer2(x.to(
            'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'
        ))
        
        # Layer 3 back on GPU 0
        x = self.layer3(x.to('cuda:0'))
        
        return x

if torch.cuda.device_count() >= 2:
    pp_model = PipelineParallelModel()
    test_input = torch.randn(32, 784)
    output = pp_model(test_input)
    print(f"✅ 流水線並行模型輸出形狀: {output.shape}")
else:
    print("⚠️ 需要至少2個GPU才能有效使用流水線並行")

## 4. 多節點訓練

### 4.1 使用torchrun啟動

In [ ]:
# 多節點訓練腳本
multi_node_script = '''
# 節點0 (主節點)
torchrun \\
    --nproc_per_node=4 \\
    --nnodes=2 \\
    --node_rank=0 \\
    --master_addr="192.168.1.1" \\
    --master_port=29500 \\
    train_ddp.py

# 節點1
torchrun \\
    --nproc_per_node=4 \\
    --nnodes=2 \\
    --node_rank=1 \\
    --master_addr="192.168.1.1" \\
    --master_port=29500 \\
    train_ddp.py
'''

print("多節點訓練命令:")
print(multi_node_script)

print("\n參數說明:")
print("  --nproc_per_node: 每個節點的GPU數量")
print("  --nnodes: 節點總數")
print("  --node_rank: 當前節點編號 (0為主節點)")
print("  --master_addr: 主節點IP地址")
print("  --master_port: 通信端口")

## 5. PyTorch Lightning - 簡化分布式訓練

PyTorch Lightning 提供了更高層次的API，自動處理分布式訓練的細節。

In [ ]:
# PyTorch Lightning 示例
lightning_code = '''
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset

class LitModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = SimpleModel()
        self.criterion = nn.CrossEntropyLoss()
    
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.log('train_loss', loss)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)

# 創建數據
fake_data = torch.randn(10000, 784)
fake_labels = torch.randint(0, 10, (10000,))
dataset = TensorDataset(fake_data, fake_labels)
dataloader = DataLoader(dataset, batch_size=128)

# 創建模型
model = LitModel()

# 創建Trainer
trainer = pl.Trainer(
    max_epochs=3,
    accelerator='gpu',
    devices=2,                    # 使用2個GPU
    strategy='ddp',               # 使用DDP策略
    precision=16,                 # 混合精度
    log_every_n_steps=50
)

# 訓練
trainer.fit(model, dataloader)
'''

with open('train_lightning.py', 'w') as f:
    f.write(lightning_code)

print("✅ PyTorch Lightning training code saved")
print("\n安裝 PyTorch Lightning:")
print("  pip install pytorch-lightning")
print("\n運行:")
print("  python train_lightning.py")

## 6. 性能優化技巧

### 6.1 梯度累積

In [ ]:
def train_with_gradient_accumulation(model, dataloader, accumulation_steps=4):
    """梯度累積 - 模擬更大的batch size"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    optimizer.zero_grad()
    
    for i, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.cuda(), labels.cuda()
        
        # 前向傳播
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # 縮放損失
        loss = loss / accumulation_steps
        loss.backward()
        
        # 每accumulation_steps步更新一次權重
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            
            if (i + 1) % 100 == 0:
                print(f"Step [{i+1}], Loss: {loss.item() * accumulation_steps:.4f}")

print("梯度累積的優點:")
print("  - 在內存有限時模擬更大的batch size")
print("  - 有效batch size = batch_size × accumulation_steps")
print("  - 節省GPU內存")

### 6.2 梯度檢查點 (Gradient Checkpointing)

In [ ]:
from torch.utils.checkpoint import checkpoint

class CheckpointedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 512)
        self.layer2 = nn.Linear(512, 256)
        self.layer3 = nn.Linear(256, 10)
    
    def forward(self, x):
        # 使用檢查點保存內存
        x = checkpoint(self._forward_layer1, x)
        x = checkpoint(self._forward_layer2, x)
        x = self.layer3(x)
        return x
    
    def _forward_layer1(self, x):
        return torch.relu(self.layer1(x))
    
    def _forward_layer2(self, x):
        return torch.relu(self.layer2(x))

print("梯度檢查點的優點:")
print("  - 減少內存使用 (犧牲一些計算時間)")
print("  - 允許訓練更大的模型")
print("  - 在反向傳播時重新計算激活值")

## 7. 監控和調試

### 7.1 分布式訓練監控

In [ ]:
# 分布式訓練監控代碼
monitoring_code = '''
import torch.distributed as dist
import time

class DistributedMonitor:
    def __init__(self, rank, world_size):
        self.rank = rank
        self.world_size = world_size
        self.start_time = time.time()
    
    def log_metrics(self, metrics):
        """同步並記錄所有進程的指標"""
        # 將指標轉換為tensor
        for key, value in metrics.items():
            tensor = torch.tensor(value).cuda()
            
            # All-reduce平均
            dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
            tensor /= self.world_size
            
            if self.rank == 0:
                print(f"{key}: {tensor.item():.4f}")
    
    def log_throughput(self, samples_processed):
        """計算並記錄吞吐量"""
        elapsed = time.time() - self.start_time
        throughput = samples_processed / elapsed
        
        if self.rank == 0:
            print(f"Throughput: {throughput:.2f} samples/sec")
'''

print(monitoring_code)

## 8. AI輔助工具

### 8.1 使用Weights & Biases追踪分布式訓練

In [ ]:
wandb_code = '''
import wandb

def train_with_wandb(rank, world_size):
    # 只在主進程初始化wandb
    if rank == 0:
        wandb.init(
            project="distributed-training",
            config={
                "learning_rate": 0.001,
                "epochs": 3,
                "batch_size": 128,
                "world_size": world_size
            }
        )
    
    # ... 訓練代碼 ...
    
    # 記錄指標
    if rank == 0:
        wandb.log({
            "loss": loss.item(),
            "epoch": epoch,
            "gpu_memory": torch.cuda.memory_allocated() / 1e9
        })
'''

print("安裝 Weights & Biases:")
print("  pip install wandb")
print("\n初始化:")
print("  wandb login")

## 小結

本章介紹了分布式訓練的核心概念和實踐：

✅ **數據並行**：DataParallel vs DistributedDataParallel  
✅ **模型並行**：張量並行、流水線並行  
✅ **混合精度**：AMP自動混合精度訓練  
✅ **多節點訓練**：使用torchrun啟動  
✅ **優化技巧**：梯度累積、梯度檢查點  
✅ **高級工具**：PyTorch Lightning、W&B監控

## 練習

1. 實現一個使用DDP的完整訓練腳本
2. 比較DataParallel和DDP的性能差異
3. 使用混合精度訓練並測量加速比
4. 實現梯度累積並驗證等效性
5. 使用PyTorch Lightning重構你的訓練代碼

## 推薦資源

- 📖 [PyTorch Distributed Tutorial](https://pytorch.org/tutorials/beginner/dist_overview.html)
- 📖 [PyTorch Lightning Documentation](https://pytorch-lightning.readthedocs.io/)
- 📖 [NVIDIA Apex](https://github.com/NVIDIA/apex)
- 📖 [DeepSpeed](https://www.deepspeed.ai/)
- 📖 [Horovod](https://horovod.readthedocs.io/)